In [2]:
# Import Required Libraries
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.orm import sessionmaker

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

from cryptoquant.config.settings import get_settings
from cryptoquant.collectors.coinbase_client import CoinbaseClient
from cryptoquant.database.models import Asset
from cryptoquant.database.session import get_session

print("✓ All imports successful")

✓ All imports successful


In [3]:
# Load settings and create database connection
settings = get_settings()
print(f"Database URL: {settings.database_url.split('@')[1]}")  # Hide credentials

# Create engine
engine = create_engine(settings.database_url)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@VERSION as version"))
        version = result.fetchone()[0]
        print(f"✓ Database connection successful!\n")
        print(f"SQL Server Version: {version[:100]}...")
        
        # Check if crypto schema exists
        result = conn.execute(text("""
            SELECT SCHEMA_NAME 
            FROM INFORMATION_SCHEMA.SCHEMATA 
            WHERE SCHEMA_NAME = 'crypto'
        """))
        schema = result.fetchone()
        print(f"\n✓ crypto schema exists: {schema is not None}")
        
        # Check if assets table exists
        result = conn.execute(text("""
            SELECT TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_SCHEMA = 'crypto' AND TABLE_NAME = 'assets'
        """))
        table = result.fetchone()
        print(f"✓ crypto.assets table exists: {table is not None}")
        
except Exception as e:
    print(f"✗ Database connection failed: {e}")
    raise

Database URL: fin-market-sqlserver.database.windows.net:1433/fin-market-db?driver=ODBC+Driver+18+for+SQL+Server
✓ Database connection successful!

SQL Server Version: Microsoft SQL Azure (RTM) - 12.0.2000.8 
	Jul  2 2026 14:28:12 
	Copyright (C) 2026 Microsoft Corpor...

✓ crypto schema exists: True
✓ crypto.assets table exists: True


In [4]:
# Initialize Coinbase client
client = CoinbaseClient(
    api_key=settings.coinbase_api_key,
    api_secret=settings.coinbase_api_secret
)

# Fetch all products
print("Fetching products from Coinbase API...")
products = client.get_products()
print(f"✓ Retrieved {len(products)} products")

# Display sample product
print("\nSample product:")
print(products[0])

Fetching products from Coinbase API...
✓ Retrieved 912 products

Sample product:
product_id='BTC-USD' base_currency_id='BTC' quote_currency_id='USD' base_display_symbol='BTC' quote_display_symbol='USD' status='online' trading_disabled=False base_increment=Decimal('1E-8') quote_increment=Decimal('0.01') base_min_size=Decimal('1E-8') base_max_size=Decimal('3400') quote_min_size=Decimal('1') quote_max_size=Decimal('150000000')


In [5]:
# Convert to DataFrame for analysis
products_df = pd.DataFrame([p.model_dump() for p in products])
print(f"Products DataFrame shape: {products_df.shape}")
print(f"\nColumns in Product API response:")
print(products_df.columns.tolist())
print(f"\nSample data:")
products_df.head()

Products DataFrame shape: (912, 13)

Columns in Product API response:
['product_id', 'base_currency_id', 'quote_currency_id', 'base_display_symbol', 'quote_display_symbol', 'status', 'trading_disabled', 'base_increment', 'quote_increment', 'base_min_size', 'base_max_size', 'quote_min_size', 'quote_max_size']

Sample data:


,product_id,base_currency_id,quote_currency_id,base_display_symbol,quote_display_symbol,status,trading_disabled,base_increment,quote_increment,base_min_size,base_max_size,quote_min_size,quote_max_size
0,BTC-USD,BTC,USD,BTC,USD,online,False,1E-8,0.01,1E-8,3400,1,150000000
1,BTC-USDC,BTC,USDC,BTC,USD,online,False,1E-8,0.01,1E-8,3400,1,150000000
2,ETH-USD,ETH,USD,ETH,USD,online,False,1E-8,0.01,1E-8,42000,1,150000000
3,ETH-USDC,ETH,USDC,ETH,USD,online,False,1E-8,0.01,1E-8,42000,1,150000000
4,XRP-USD,XRP,USD,XRP,USD,online,False,0.000001,0.0001,0.000001,11996772.8407292192545075,1,10000000


In [6]:
# Extract unique currencies with product_id from base and quote currencies
base_currencies = products_df['base_currency_id'].unique()
quote_currencies = products_df['quote_currency_id'].unique()
all_currencies = sorted(set(list(base_currencies) + list(quote_currencies)))

print(f"✓ Found {len(all_currencies)} unique currencies")

# Create assets DataFrame with product_id and display_symbol
assets_data = []
for currency in all_currencies:
    as_base = products_df[products_df['base_currency_id'] == currency]
    
    if len(as_base) > 0:
        product = as_base.iloc[0]
        assets_data.append({
            'product_id': product['product_id'],
            'symbol': currency,
            'name': currency,
            'display_symbol': product.get('base_display_symbol', currency),
            'asset_type': 'fiat' if currency in ['USD', 'EUR', 'GBP', 'USDT', 'USDC'] else 'crypto'
        })
    else:
        as_quote = products_df[products_df['quote_currency_id'] == currency]
        if len(as_quote) > 0:
            product = as_quote.iloc[0]
            assets_data.append({
                'product_id': product['product_id'],
                'symbol': currency,
                'name': currency,
                'display_symbol': product.get('quote_display_symbol', currency),
                'asset_type': 'fiat' if currency in ['USD', 'EUR', 'GBP', 'USDT', 'USDC'] else 'crypto'
            })

assets_df = pd.DataFrame(assets_data)
print(f"Assets extracted: {len(assets_df)}")
print(f"Columns: {assets_df.columns.tolist()}")
assets_df.head(10)

✓ Found 409 unique currencies
Assets extracted: 409
Columns: ['product_id', 'symbol', 'name', 'display_symbol', 'asset_type']


,product_id,symbol,name,display_symbol,asset_type
0,00-USD,00,00,00,crypto
1,1INCH-USD,1INCH,1INCH,1INCH,crypto
2,2Z-USD,2Z,2Z,2Z,crypto
3,A8-USD,A8,A8,A8,crypto
4,AAVE-USD,AAVE,AAVE,AAVE,crypto
5,ABT-USD,ABT,ABT,ABT,crypto
6,ACH-USD,ACH,ACH,ACH,crypto
7,ACS-USD,ACS,ACS,ACS,crypto
8,ADA-USD,ADA,ADA,ADA,crypto
9,AERGO-USD,AERGO,AERGO,AERGO,crypto


In [7]:
# Verify data load
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM crypto.assets"))
    print(f"Total assets: {result.fetchone()[0]}")
    
    result = conn.execute(text("""
        SELECT TOP 20 id, product_id, symbol, display_symbol, asset_type 
        FROM crypto.assets 
        ORDER BY symbol
    """))
    
    records = result.fetchall()
    df = pd.DataFrame(records, columns=['id', 'product_id', 'symbol', 'display_symbol', 'asset_type'])
    display(df)
    
    result = conn.execute(text("""
        SELECT asset_type, COUNT(*) as count 
        FROM crypto.assets 
        GROUP BY asset_type
        ORDER BY count DESC
    """))
    
    print("\nAsset type distribution:")
    for asset_type, count in result.fetchall():
        print(f"  {asset_type}: {count}")

Total assets: 409


,id,product_id,symbol,display_symbol,asset_type
0,1228,00-USD,00,00,crypto
1,1229,1INCH-USD,1INCH,1INCH,crypto
2,1230,2Z-USD,2Z,2Z,crypto
3,1231,A8-USD,A8,A8,crypto
4,1232,AAVE-USD,AAVE,AAVE,crypto
5,1233,ABT-USD,ABT,ABT,crypto
6,1234,ACH-USD,ACH,ACH,crypto
7,1235,ACS-USD,ACS,ACS,crypto
8,1236,ADA-USD,ADA,ADA,crypto
9,1237,AERGO-USD,AERGO,AERGO,crypto



Asset type distribution:
  crypto: 404
  fiat: 5


In [8]:
# Reimport Asset model to get updated schema
import importlib
import sys
if 'cryptoquant.database.models' in sys.modules:
    importlib.reload(sys.modules['cryptoquant.database.models'])
from cryptoquant.database.models import Asset

# Clear and reload with product_id
with engine.connect() as conn:
    conn.execute(text("DELETE FROM crypto.assets"))
    conn.commit()
    print("✓ Table cleared")

Session = sessionmaker(bind=engine)
session = Session()

try:
    for _, row in assets_df.iterrows():
        new_asset = Asset(
            product_id=row['product_id'],
            symbol=row['symbol'],
            name=row['name'],
            display_symbol=row.get('display_symbol', row['symbol']),
            asset_type=row.get('asset_type', 'crypto')
        )
        session.add(new_asset)
    
    session.commit()
    print(f"✓ Loaded {len(assets_df)} assets with product_id")
    
except Exception as e:
    session.rollback()
    print(f"✗ Error: {e}")
    raise
finally:
    session.close()

# Verify
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT TOP 10 id, product_id, symbol, display_symbol, asset_type 
        FROM crypto.assets 
        WHERE symbol IN ('BTC', 'ETH', 'USD', 'USDC', 'SOL', 'ADA', 'XRP')
        ORDER BY symbol
    """))
    
    df_verify = pd.DataFrame(result.fetchall(), columns=['id', 'product_id', 'symbol', 'display_symbol', 'asset_type'])
    print("\n✓ Verification:")
    display(df_verify)

IntegrityError: (pyodbc.IntegrityError) ('23000', '[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The DELETE statement conflicted with the REFERENCE constraint "FK__trading_p__base___05D8E0BE". The conflict occurred in database "fin-market-db", table "crypto.trading_pairs", column \'base_asset_id\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]The statement has been terminated. (3621)')
[SQL: DELETE FROM crypto.assets]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [ ]:
# Final schema verification
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE, ORDINAL_POSITION
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = 'crypto' AND TABLE_NAME = 'assets'
        ORDER BY ORDINAL_POSITION
    """))
    
    print("crypto.assets table schema:")
    for row in result.fetchall():
        print(f"  {row.ORDINAL_POSITION}. {row.COLUMN_NAME} ({row.DATA_TYPE}) - Nullable: {row.IS_NULLABLE}")

crypto.assets table schema:
  1. id (int) - Nullable: NO
  2. symbol (varchar) - Nullable: NO
  3. name (varchar) - Nullable: NO
  4. display_symbol (varchar) - Nullable: YES
  5. asset_type (varchar) - Nullable: NO
  6. decimals (int) - Nullable: NO
  7. active (bit) - Nullable: NO
  8. created_at (datetime) - Nullable: NO
  9. updated_at (datetime) - Nullable: NO
  10. product_id (varchar) - Nullable: YES
